In [19]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions
import utilities.plot as plot

# Reload the module to reflect the changes
importlib.reload(functions)
importlib.reload(plot)

<module 'utilities.plot' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/plot.py'>

In [20]:
from collections import Counter

IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()
# print("IN consensus sequence:", IN_consensus_seq)
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# print(IN_consensus_seq)
    
IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

# Consensus from unreduced IN MSA: most common residue at each position
seq_len = len(IN_all_seq_unreduced[0])
# assert all(len(s) == seq_len for s in IN_all_seq_unreduced), "All IN sequences must have equal length."

IN_consensus_unreduced = ''.join(
    Counter(seq[i] for seq in IN_all_seq_unreduced).most_common(1)[0][0]
    for i in range(seq_len)
)

# Optional quick check
# print(IN_consensus_unreduced)

In [21]:
IN_weights_path = 'IN/data/in.weights.txt'
len_IN_all_seqs = len(IN_all_seq)
with open(IN_weights_path, 'r') as f:
    IN_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(IN_weights) == len_IN_all_seqs, "Weights and sequences must have the same length."

IN_pairs = [
    'G140S-Q148H',
    'Y143C-S230R',
    'G140A-Q148K',
    'G140S-Q148R',
    'G140S-Q148K',
    'G140A-Q148R',
    'E138K-Q148K',
    'G140A-Q148H',
    'E138K-Q148R',
    'Y143C-S230K',
    'N155H-E170A',
    'E138K-S147G',
    'S147G-Q148R',
    'Y143R-I151V',
    'E138K-Q148H',
    'S147G-L158V',
    'E92Q-K215R',
    'E138A-Q148H',
    'E138K-S230K',
    'E138K-Y194C',
]


In [24]:
seq_mutation_index = {}
for pair in IN_pairs:
    mut1,mut2 = functions.split_pairs(pair)
    # print(f"Processing pair: {pair}, mut1: {mut1}, mut2: {mut2}")
    for i, seq in enumerate(IN_all_seq_unreduced):
        mutations = functions.get_mutations_from_sequence(IN_consensus_unreduced, seq, start_index=1)
        if mut1 in mutations and mut2 in mutations:
            if pair not in seq_mutation_index:
                seq_mutation_index[pair] = []
            seq_mutation_index[pair].append(i)
            # print(f"Sequence {i} has both mutations: {mut1}, {mut2}")

for pair, indices in seq_mutation_index.items():
    pair1, pair2 = functions.split_pairs(pair)
    pair1_reduced = functions.unreduced_to_reduced(IN_redux, pair1)
    pair2_reduced = functions.unreduced_to_reduced(IN_redux, pair2)
    # print(pair1_reduced, pair2_reduced)
    total_mut = len(indices)
    gof_count = 0
    for index in indices:
        if functions.is_gof_seq(IN_all_seq[index], pair1_reduced, pair2_reduced,IN_J,1,263):
            gof_count += 1
    print(f"Pair {pair} has {gof_count} GOF sequences out of {total_mut}, fraction: {gof_count/total_mut if total_mut > 0 else 0}")
        

Pair G140S-Q148H has 122 GOF sequences out of 221, fraction: 0.5520361990950227
Pair Y143C-S230R has 16 GOF sequences out of 22, fraction: 0.7272727272727273
Pair G140A-Q148K has 4 GOF sequences out of 4, fraction: 1.0
Pair G140S-Q148R has 8 GOF sequences out of 19, fraction: 0.42105263157894735
Pair G140A-Q148R has 7 GOF sequences out of 10, fraction: 0.7
Pair E138K-Q148K has 3 GOF sequences out of 4, fraction: 0.75
Pair E138K-Q148R has 17 GOF sequences out of 21, fraction: 0.8095238095238095
Pair Y143C-S230K has 1 GOF sequences out of 1, fraction: 1.0
Pair N155H-E170A has 4 GOF sequences out of 5, fraction: 0.8
Pair E138K-S147G has 10 GOF sequences out of 13, fraction: 0.7692307692307693
Pair S147G-Q148R has 11 GOF sequences out of 11, fraction: 1.0
Pair E138K-Q148H has 8 GOF sequences out of 10, fraction: 0.8
Pair S147G-L158V has 1 GOF sequences out of 1, fraction: 1.0
Pair E92Q-K215R has 1 GOF sequences out of 1, fraction: 1.0
Pair E138A-Q148H has 9 GOF sequences out of 15, fractio